# K-Nearest Neighbors
# Euclidean, Manhattan, Minkowski distances
# Scaling coordinates
# Cosine similarity / distance
# Gradient boost

# http://vision.stanford.edu/teaching/cs231n-demos/knn/
# https://mmcbrien.com/writing_dir/InteractiveNeighbors.html
# https://antoinebrl.github.io/blog/knn-variance/

In [ ]:
%pip install mplcursors

In [ ]:
%matplotlib widget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mplcursors

from ipywidgets import interact, FloatSlider, IntSlider, Dropdown


# -----------------------------
# 1. Create sample data on a 2D grid
# -----------------------------

np.random.seed(42)

x_values = np.linspace(0, 10, 11)
y_values = np.linspace(0, 10, 11)

xx, yy = np.meshgrid(x_values, y_values)

X = np.column_stack([xx.ravel(), yy.ravel()])


def true_function(x, y):
    return np.sin(x / 2) + np.cos(y / 2)


y = true_function(X[:, 0], X[:, 1]) + np.random.normal(0, 0.15, size=X.shape[0])


# -----------------------------
# 2. Distance functions
# -----------------------------

def compute_distances(X, new_sample, metric="euclidean", p=2):
    dx = np.abs(X[:, 0] - new_sample[0])
    dy = np.abs(X[:, 1] - new_sample[1])

    if metric == "euclidean":
        distances = np.sqrt(dx**2 + dy**2)

    elif metric == "manhattan":
        distances = dx + dy

    elif metric == "minkowski":
        distances = (dx**p + dy**p) ** (1 / p)

    else:
        raise ValueError("metric must be 'euclidean', 'manhattan', or 'minkowski'")

    return distances


def metric_formula_text(metric, p):
    if metric == "euclidean":
        return r"Euclidean distance: d_i = sqrt((x_i - x_new)^2 + (y_i - y_new)^2)"

    if metric == "manhattan":
        return r"Manhattan distance: d_i = |x_i - x_new| + |y_i - y_new|"

    if metric == "minkowski":
        return (
            f"Minkowski distance with p = {p:.2f}: "
            r"d_i = (|x_i - x_new|^p + |y_i - y_new|^p)^(1/p)"
        )


def prediction_formula_text(weights):
    if weights == "uniform":
        return (
            "Uniform KNN prediction:\n"
            "y_hat = (1 / k) * sum(y_i for i in nearest neighbors)"
        )

    if weights == "distance":
        return (
            "Distance-weighted KNN prediction:\n"
            "w_i = 1 / d_i\n"
            "y_hat = sum(w_i * y_i) / sum(w_i)\n"
            "If d_i = 0 for one or more neighbors, only the zero-distance neighbors are averaged."
        )


# -----------------------------
# 3. Interactive KNN prediction
# -----------------------------

def plot_knn_prediction(
    new_x=5.0,
    new_y=5.0,
    k=5,
    weights="uniform",
    metric="euclidean",
    minkowski_p=2.0
):
    plt.close("all")

    new_sample = np.array([new_x, new_y])

    if metric == "euclidean":
        effective_p = 2
    elif metric == "manhattan":
        effective_p = 1
    else:
        effective_p = minkowski_p

    distances = compute_distances(
        X=X,
        new_sample=new_sample,
        metric=metric,
        p=effective_p
    )

    neighbor_indices = np.argsort(distances)[:k]
    neighbor_distances = distances[neighbor_indices]
    neighbor_values = y[neighbor_indices]

    if weights == "uniform":
        predicted_value = np.mean(neighbor_values)
        neighbor_weights = np.ones_like(neighbor_values) / k

    elif weights == "distance":
        zero_distance_mask = neighbor_distances == 0

        if np.any(zero_distance_mask):
            zero_distance_values = neighbor_values[zero_distance_mask]
            predicted_value = np.mean(zero_distance_values)

            neighbor_weights = np.zeros_like(neighbor_values)
            neighbor_weights[zero_distance_mask] = 1 / zero_distance_mask.sum()

        else:
            raw_weights = 1 / neighbor_distances
            neighbor_weights = raw_weights / raw_weights.sum()
            predicted_value = np.sum(neighbor_weights * neighbor_values)

    # Plot
    fig, ax = plt.subplots(figsize=(9, 8))

    scatter = ax.scatter(
        X[:, 0],
        X[:, 1],
        c=y,
        cmap="viridis",
        s=80,
        edgecolor="black",
        label="Training samples"
    )

    fig.colorbar(scatter, ax=ax, label="Sample value")

    neighbor_scatter = ax.scatter(
        X[neighbor_indices, 0],
        X[neighbor_indices, 1],
        s=230,
        facecolors="none",
        edgecolors="red",
        linewidths=2.5,
        label=f"{k} nearest neighbors"
    )

    new_scatter = ax.scatter(
        new_x,
        new_y,
        c="orange",
        s=270,
        marker="*",
        edgecolor="black",
        linewidths=1.5,
        label="New sample"
    )

    for idx in neighbor_indices:
        ax.plot(
            [new_x, X[idx, 0]],
            [new_y, X[idx, 1]],
            color="red",
            linestyle="--",
            alpha=0.45
        )

    ax.set_title(
        f"KNN Regression Prediction\n"
        f"New sample = ({new_x:.2f}, {new_y:.2f}), "
        f"k = {k}, weights = {weights}, metric = {metric}\n"
        f"Predicted value = {predicted_value:.4f}"
    )

    ax.set_xlabel("x coordinate")
    ax.set_ylabel("y coordinate")

    ax.set_xlim(-0.5, 10.5)
    ax.set_ylim(-0.5, 10.5)
    ax.set_aspect("equal", adjustable="box")

    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")

    # --------------------------------------------------
    # Hover labels for training points
    # --------------------------------------------------

    cursor = mplcursors.cursor(scatter, hover=True)

    @cursor.connect("add")
    def on_add(sel):
        i = sel.index

        sel.annotation.set_text(
            f"point index: {i}\n"
            f"x = {X[i, 0]:.2f}\n"
            f"y = {X[i, 1]:.2f}\n"
            f"value = {y[i]:.4f}\n"
            f"distance to new sample = {distances[i]:.4f}"
        )

        sel.annotation.get_bbox_patch().set(
            facecolor="white",
            alpha=0.95
        )

    plt.show()

    # Print formulas
    print("=" * 70)
    print("DISTANCE FORMULA")
    print("=" * 70)
    print(metric_formula_text(metric, effective_p))
    print()

    print("=" * 70)
    print("PREDICTION FORMULA")
    print("=" * 70)
    print(prediction_formula_text(weights))
    print()

    print("=" * 70)
    print("NUMERICAL CALCULATION")
    print("=" * 70)

    print(f"New sample: ({new_x:.4f}, {new_y:.4f})")
    print(f"k: {k}")
    print(f"Weighting: {weights}")
    print(f"Distance metric: {metric}")

    if metric == "minkowski":
        print(f"Minkowski p: {effective_p:.4f}")

    print()
    print("Nearest neighbors:")

    for rank, idx in enumerate(neighbor_indices, start=1):
        xi, yi = X[idx]
        value = y[idx]
        distance = distances[idx]
        weight = neighbor_weights[rank - 1]

        print(
            f"{rank}. point=({xi:.2f}, {yi:.2f}), "
            f"value={value:.4f}, "
            f"distance={distance:.4f}, "
            f"normalized_weight={weight:.4f}"
        )

    print()

    if weights == "uniform":
        value_terms = " + ".join([f"{v:.4f}" for v in neighbor_values])
        print("Uniform average calculation:")
        print(f"y_hat = ({value_terms}) / {k}")
        print(f"y_hat = {predicted_value:.4f}")

    elif weights == "distance":
        zero_distance_mask = neighbor_distances == 0

        if np.any(zero_distance_mask):
            zero_values = neighbor_values[zero_distance_mask]
            value_terms = " + ".join([f"{v:.4f}" for v in zero_values])

            print("Zero-distance case:")
            print("At least one neighbor has distance 0.")
            print("The prediction is the average of the zero-distance neighbors only.")
            print(f"y_hat = ({value_terms}) / {len(zero_values)}")
            print(f"y_hat = {predicted_value:.4f}")

        else:
            weighted_terms = [
                f"({neighbor_weights[i]:.4f} * {neighbor_values[i]:.4f})"
                for i in range(k)
            ]

            print("Distance-weighted average calculation:")
            print(
                "Raw weights before normalization: "
                + ", ".join([f"1/{d:.4f}" for d in neighbor_distances])
            )
            print(
                "Normalized weights: "
                + ", ".join([f"{w:.4f}" for w in neighbor_weights])
            )
            print("y_hat = " + " + ".join(weighted_terms))
            print(f"y_hat = {predicted_value:.4f}")


# -----------------------------
# 4. Jupyter interactive controls
# -----------------------------

interact(
    plot_knn_prediction,
    new_x=FloatSlider(
        value=5.0,
        min=0.0,
        max=10.0,
        step=0.1,
        description="New x"
    ),
    new_y=FloatSlider(
        value=5.0,
        min=0.0,
        max=10.0,
        step=0.1,
        description="New y"
    ),
    k=IntSlider(
        value=5,
        min=1,
        max=20,
        step=1,
        description="k"
    ),
    weights=Dropdown(
        options=["uniform", "distance"],
        value="uniform",
        description="Weights"
    ),
    metric=Dropdown(
        options=["euclidean", "manhattan", "minkowski"],
        value="euclidean",
        description="Distance"
    ),
    minkowski_p=FloatSlider(
        value=2.0,
        min=1.0,
        max=10.0,
        step=0.1,
        description="Minkowski p"
    )
)

In [ ]:
p = 1
arr1 = np.array([0, 0])
arr2 = np.array([3, 4])
(np.sum(np.abs(arr1 - arr2)**p))**(1/p)

In [ ]:
(np.abs(0 - 3)**0.2 + np.abs(0 - 4)**0.2)**5

In [ ]:
p = 0.2
arr1 = np.array([1, 1])
arr2 = np.array([2, 4])
(np.sum(np.abs(arr1 - arr2)**p))**(1/p)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier
from ipywidgets import interact, Dropdown, IntSlider, FloatSlider


def knn_decision_boundary(
    k=7,
    metric="euclidean",
    minkowski_p=2.0,
    weights="uniform",
    noise=0.20,
    x_scale=1.0,
    y_scale=1.0,
    query_x=0.8,
    query_y=0.2
):
    # Recreate the same underlying dataset each time.
    X, y = make_moons(
        n_samples=220,
        noise=noise,
        random_state=7
    )

    # Scaling changes the distance calculation, not the displayed coordinates.
    scale = np.array([x_scale, y_scale])
    X_scaled = X * scale

    if metric == "euclidean":
        p_effective = 2
    elif metric == "manhattan":
        p_effective = 1
    else:
        p_effective = minkowski_p

    model = KNeighborsClassifier(
        n_neighbors=k,
        weights=weights,
        metric="minkowski",
        p=p_effective
    )
    model.fit(X_scaled, y)

    # Evaluate the model across a grid to reveal its decision regions.
    x_min, x_max = -1.5, 2.5
    y_min, y_max = -1.2, 1.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 220),
        np.linspace(y_min, y_max, 200)
    )

    grid = np.column_stack([xx.ravel(), yy.ravel()])
    grid_scaled = grid * scale

    probability_class_1 = model.predict_proba(grid_scaled)[:, 1]
    probability_class_1 = probability_class_1.reshape(xx.shape)

    # Find the query point's nearest neighbors using the same scaled metric.
    query = np.array([[query_x, query_y]])
    query_scaled = query * scale

    neighbor_distances, neighbor_indices = model.kneighbors(query_scaled)
    neighbor_indices = neighbor_indices[0]
    neighbor_distances = neighbor_distances[0]

    query_probability = model.predict_proba(query_scaled)[0, 1]
    query_prediction = model.predict(query_scaled)[0]

    # Plot probability regions, training data, query point, and its neighbors.
    fig, ax = plt.subplots(figsize=(10, 7))

    probability_plot = ax.contourf(
        xx,
        yy,
        probability_class_1,
        levels=np.linspace(0, 1, 21),
        cmap="RdBu_r",
        alpha=0.38,
        extend="both"
    )

    ax.contour(
        xx,
        yy,
        probability_class_1,
        levels=[0.5],
        colors="black",
        linewidths=2
    )

    ax.scatter(
        X[y == 0, 0],
        X[y == 0, 1],
        color="#2878B5",
        edgecolor="white",
        linewidth=0.7,
        s=55,
        label="Class 0"
    )

    ax.scatter(
        X[y == 1, 0],
        X[y == 1, 1],
        color="#E07A1F",
        edgecolor="white",
        linewidth=0.7,
        s=55,
        label="Class 1"
    )

    # Circle the selected neighbors and connect them to the query point.
    ax.scatter(
        X[neighbor_indices, 0],
        X[neighbor_indices, 1],
        s=210,
        facecolors="none",
        edgecolors="lime",
        linewidths=2.5,
        label=f"{k} nearest neighbors"
    )

    for idx in neighbor_indices:
        ax.plot(
            [query_x, X[idx, 0]],
            [query_y, X[idx, 1]],
            color="lime",
            linestyle="--",
            linewidth=1.2,
            alpha=0.65
        )

    ax.scatter(
        query_x,
        query_y,
        marker="*",
        s=350,
        color="yellow",
        edgecolor="black",
        linewidth=1.5,
        zorder=5,
        label="Query point"
    )

    colorbar = fig.colorbar(probability_plot, ax=ax)
    colorbar.set_label("Predicted probability of class 1")

    metric_description = metric
    if metric == "minkowski":
        metric_description += f" (p={p_effective:.1f})"

    ax.set_title(
        f"KNN decision boundary | k={k}, {metric_description}, "
        f"weights={weights}\n"
        f"Scales: x={x_scale:.2f}, y={y_scale:.2f} | "
        f"Query prediction: class {query_prediction}, "
        f"P(class 1)={query_probability:.2f}"
    )

    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)
    ax.legend(loc="upper right")
    plt.show()

    print(
        f"Query point: ({query_x:.2f}, {query_y:.2f}) | "
        f"Predicted class: {query_prediction} | "
        f"P(class 1): {query_probability:.3f}"
    )
    print("Nearest neighbors:")

    for rank, (idx, distance) in enumerate(
        zip(neighbor_indices, neighbor_distances),
        start=1
    ):
        print(
            f"{rank}. point=({X[idx, 0]:.2f}, {X[idx, 1]:.2f}), "
            f"class={y[idx]}, distance={distance:.3f}"
        )


interact(
    knn_decision_boundary,
    k=IntSlider(
        value=7,
        min=1,
        max=25,
        step=2,
        description="k"
    ),
    metric=Dropdown(
        options=["euclidean", "manhattan", "minkowski"],
        value="euclidean",
        description="Metric"
    ),
    minkowski_p=FloatSlider(
        value=2.0,
        min=1.0,
        max=8.0,
        step=0.1,
        description="Minkowski p"
    ),
    weights=Dropdown(
        options=["uniform", "distance"],
        value="uniform",
        description="Weights"
    ),
    noise=FloatSlider(
        value=0.20,
        min=0.05,
        max=0.45,
        step=0.05,
        description="Noise"
    ),
    x_scale=FloatSlider(
        value=1.0,
        min=0.3,
        max=3.0,
        step=0.1,
        description="x scale"
    ),
    y_scale=FloatSlider(
        value=1.0,
        min=0.3,
        max=3.0,
        step=0.1,
        description="y scale"
    ),
    query_x=FloatSlider(
        value=0.8,
        min=-1.5,
        max=2.5,
        step=0.05,
        description="Query x"
    ),
    query_y=FloatSlider(
        value=0.2,
        min=-1.2,
        max=1.5,
        step=0.05,
        description="Query y"
    )
);

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact, FloatSlider, IntSlider


np.random.seed(42)


# --------------------------------------------------
# 1. Original sample points
# --------------------------------------------------

x_values = np.arange(0, 6)
y_values = np.arange(0, 6)

xx, yy = np.meshgrid(x_values, y_values)

X_original = np.column_stack([xx.ravel(), yy.ravel()])


def true_function(x, y):
    return x + y


sample_values = true_function(X_original[:, 0], X_original[:, 1])


# New sample to predict
new_sample_original = np.array([2.4, 3.2])


# --------------------------------------------------
# 2. Simple interactive scaled-distance KNN demo
# --------------------------------------------------

def scaled_knn_demo(x_scale=2.0, y_scale=1.0, k=3):
    """
    x_scale and y_scale stretch or shrink the plotted coordinates.

    Larger scale means larger visual spacing and larger contribution
    to distance.
    """

    # Make sure k is an integer
    k = int(k)

    # Scale coordinates
    X_scaled = X_original.copy().astype(float)
    X_scaled[:, 0] = X_scaled[:, 0] * x_scale
    X_scaled[:, 1] = X_scaled[:, 1] * y_scale

    new_sample_scaled = new_sample_original.copy().astype(float)
    new_sample_scaled[0] = new_sample_scaled[0] * x_scale
    new_sample_scaled[1] = new_sample_scaled[1] * y_scale

    # Euclidean distance after scaling
    distances = np.sqrt(
        (X_scaled[:, 0] - new_sample_scaled[0]) ** 2
        + (X_scaled[:, 1] - new_sample_scaled[1]) ** 2
    )

    neighbor_indices = np.argsort(distances)[:k]

    prediction = np.mean(sample_values[neighbor_indices])

    # Plot
    plt.figure(figsize=(9, 6))

    scatter = plt.scatter(
        X_scaled[:, 0],
        X_scaled[:, 1],
        c=sample_values,
        cmap="viridis",
        s=90,
        edgecolor="black",
        label="Known samples"
    )

    plt.colorbar(scatter, label="Sample value")

    plt.scatter(
        X_scaled[neighbor_indices, 0],
        X_scaled[neighbor_indices, 1],
        s=260,
        facecolors="none",
        edgecolors="red",
        linewidths=2.5,
        label=f"{k} nearest neighbors"
    )

    plt.scatter(
        new_sample_scaled[0],
        new_sample_scaled[1],
        c="orange",
        marker="*",
        s=300,
        edgecolor="black",
        label="New sample"
    )

    for idx in neighbor_indices:
        plt.plot(
            [new_sample_scaled[0], X_scaled[idx, 0]],
            [new_sample_scaled[1], X_scaled[idx, 1]],
            color="red",
            linestyle="--",
            alpha=0.5
        )

    plt.title(
        "KNN with Scaled Coordinates\n"
        f"x scale = {x_scale:.2f}, y scale = {y_scale:.2f}, "
        f"prediction = {prediction:.2f}"
    )

    plt.xlabel("scaled x coordinate")
    plt.ylabel("scaled y coordinate")

    plt.axis("equal")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print("Distance formula after scaling:")
    print("distance = sqrt((x_scale * Δx)^2 + (y_scale * Δy)^2)")
    print()

    print("Original new sample:")
    print(f"({new_sample_original[0]:.2f}, {new_sample_original[1]:.2f})")
    print()

    print("Scaled new sample:")
    print(f"({new_sample_scaled[0]:.2f}, {new_sample_scaled[1]:.2f})")
    print()

    print("Nearest neighbors:")
    for idx in neighbor_indices:
        original_x, original_y = X_original[idx]
        scaled_x, scaled_y = X_scaled[idx]

        print(
            f"original=({original_x:.0f}, {original_y:.0f}), "
            f"scaled=({scaled_x:.2f}, {scaled_y:.2f}), "
            f"value={sample_values[idx]:.2f}, "
            f"distance={distances[idx]:.3f}"
        )

    print()
    print(f"KNN prediction = average neighbor value = {prediction:.2f}")


# --------------------------------------------------
# 3. Jupyter sliders
# --------------------------------------------------

interact(
    scaled_knn_demo,
    x_scale=FloatSlider(
        value=2.0,
        min=0.2,
        max=5.0,
        step=0.1,
        description="x scale"
    ),
    y_scale=FloatSlider(
        value=1.0,
        min=0.2,
        max=5.0,
        step=0.1,
        description="y scale"
    ),
    k=IntSlider(
        value=3,
        min=1,
        max=10,
        step=1,
        description="k"
    )
)

# Why KNN can struggle in high dimensions

## KNN relies on distance: it predicts using the training examples that are nearest to a new point. But as the number of dimensions grows, distances can become less informative: even the nearest and farthest points may be at fairly similar distances. This phenomenon is one aspect of the **curse of dimensionality**.

## In this demo, the points are random Gaussian vectors and the query point is the origin. The **Dimensions** slider changes how many coordinates contribute to each distance. The left plot shows the distances at the selected dimension, divided by their median; the right plot shows how the relative spread of distances and the nearest-to-farthest distance ratio change as dimensions increase.

## The demo illustrates a *potential problem for distance-based methods such as KNN*: if distances are hard to distinguish, “nearest” neighbors may not be meaningfully nearer than other points. It is not a complete demonstration of the curse of dimensionality, nor does it show that KNN always fails in high dimensions. In practice, performance depends on the data structure, the features, the distance measure, and the amount of data.

In [ ]:
"""
WHERE THIS CELL GOES:
Add it as a new cell immediately after the staged gradient-boosting demo
(the cell containing staged_gradient_boosting_demo). Leave both earlier
demos unchanged.
"""

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider


def distance_concentration_demo(dimension=20, n_points=500):
    rng = np.random.default_rng(42)

    max_dimension = 100
    points = rng.normal(size=(n_points, max_dimension))

    squared_distances = np.cumsum(points**2, axis=1)
    dimensions = np.arange(1, max_dimension + 1)
    all_distances = np.sqrt(squared_distances)

    distances = all_distances[:, dimension - 1]
    median_distance = np.median(distances)
    relative_distances = distances / median_distance

    coefficients_of_variation = np.std(all_distances, axis=0) / np.mean(
        all_distances, axis=0
    )
    nearest_to_farthest_ratios = (
        np.min(all_distances, axis=0) / np.max(all_distances, axis=0)
    )

    fig, (ax_hist, ax_trend) = plt.subplots(1, 2, figsize=(13, 5))

    ax_hist.hist(
        relative_distances,
        bins=35,
        color="#2878B5",
        edgecolor="white",
        alpha=0.85
    )
    ax_hist.axvline(
        1,
        color="black",
        linestyle="--",
        linewidth=2,
        label="Median distance"
    )
    ax_hist.axvline(
        np.min(relative_distances),
        color="#D62728",
        linewidth=2,
        label="Nearest point"
    )
    ax_hist.axvline(
        np.max(relative_distances),
        color="#2CA02C",
        linewidth=2,
        label="Farthest point"
    )
    ax_hist.set_title(
        f"Distances from the query point at dimension {dimension}\n"
        "Distances are divided by their median"
    )
    ax_hist.set_xlabel("Distance / median distance")
    ax_hist.set_ylabel("Number of points")
    ax_hist.grid(alpha=0.2)
    ax_hist.legend()

    ax_trend.plot(
        dimensions,
        coefficients_of_variation,
        color="#2878B5",
        linewidth=2.5,
        label="Relative spread: standard deviation / mean"
    )
    ax_trend.plot(
        dimensions,
        nearest_to_farthest_ratios,
        color="#D62728",
        linewidth=2.5,
        label="Nearest distance / farthest distance"
    )
    ax_trend.axvline(
        dimension,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"Selected dimension: {dimension}"
    )
    ax_trend.set_title("Distances become more alike as dimension grows")
    ax_trend.set_xlabel("Number of dimensions")
    ax_trend.set_ylabel("Ratio")
    ax_trend.set_ylim(0, 1.05)
    ax_trend.grid(alpha=0.25)
    ax_trend.legend(fontsize=9)

    fig.suptitle(
        f"{n_points} Gaussian points; query point is the origin",
        fontsize=13
    )
    plt.tight_layout()
    plt.show()

    print(f"Dimension: {dimension}")
    print(f"Nearest distance: {distances.min():.3f}")
    print(f"Median distance:  {median_distance:.3f}")
    print(f"Farthest distance: {distances.max():.3f}")
    print(
        "Nearest / farthest distance: "
        f"{distances.min() / distances.max():.3f}"
    )
    print(
        "Relative spread (standard deviation / mean): "
        f"{np.std(distances) / np.mean(distances):.3f}"
    )


interact(
    distance_concentration_demo,
    dimension=IntSlider(
        value=20,
        min=1,
        max=100,
        step=1,
        description="Dimensions"
    ),
    n_points=IntSlider(
        value=500,
        min=100,
        max=2000,
        step=100,
        description="Points"
    )
);

In [ ]:
import numpy as np
import re

from collections import Counter
from ipywidgets import Textarea, VBox, HBox, Output, Layout
from IPython.display import display, Markdown, clear_output


def tokenize(text):
    text = text.lower()
    tokens = re.findall(r"\b\w+\b", text)
    return tokens


def vectorize(doc1, doc2):
    tokens1 = tokenize(doc1)
    tokens2 = tokenize(doc2)

    vocabulary = sorted(set(tokens1) | set(tokens2))

    counts1 = Counter(tokens1)
    counts2 = Counter(tokens2)

    vector1 = np.array([counts1[word] for word in vocabulary], dtype=float)
    vector2 = np.array([counts2[word] for word in vocabulary], dtype=float)

    return vocabulary, vector1, vector2


def cosine_similarity(vector1, vector2):
    dot_product = np.dot(vector1, vector2)
    norm1 = np.linalg.norm(vector1)
    norm2 = np.linalg.norm(vector2)

    if norm1 == 0 or norm2 == 0:
        return np.nan, dot_product, norm1, norm2

    similarity = dot_product / (norm1 * norm2)
    return similarity, dot_product, norm1, norm2


doc1_box = Textarea(
    value="cat sat mat",
    placeholder="Type the first document here",
    description="Document 1:",
    layout=Layout(width="500px", height="100px")
)

doc2_box = Textarea(
    value="cat sat rug",
    placeholder="Type the second document here",
    description="Document 2:",
    layout=Layout(width="500px", height="100px")
)

output = Output()


def update_similarity(change=None):
    with output:
        clear_output(wait=True)

        doc1 = doc1_box.value
        doc2 = doc2_box.value

        vocabulary, vector1, vector2 = vectorize(doc1, doc2)

        similarity, dot_product, norm1, norm2 = cosine_similarity(vector1, vector2)

        if np.isnan(similarity):
            distance = np.nan
        else:
            distance = 1 - similarity

        display(Markdown("## Cosine Similarity and Cosine Distance"))

        display(Markdown("### Math equation"))

        display(Markdown(
            r"""
Cosine similarity compares two vectors by measuring the cosine of the angle between them.

$$
\text{cosine similarity} =
\cos(\theta)
=
\frac{\mathbf{a} \cdot \mathbf{b}}
{\|\mathbf{a}\| \|\mathbf{b}\|}
$$

where

$$
\mathbf{a} \cdot \mathbf{b}
=
\sum_{i=1}^{n} a_i b_i
$$

and

$$
\|\mathbf{a}\|
=
\sqrt{\sum_{i=1}^{n} a_i^2}
$$

Cosine distance is:

$$
\text{cosine distance} = 1 - \text{cosine similarity}
$$
"""
        ))

        display(Markdown("### Tokenization"))

        display(Markdown(f"**Document 1 tokens:** `{tokenize(doc1)}`"))
        display(Markdown(f"**Document 2 tokens:** `{tokenize(doc2)}`"))

        display(Markdown("### Vocabulary"))

        display(Markdown(f"`{vocabulary}`"))

        display(Markdown("### Count vectors"))

        display(Markdown(f"**Document 1 vector:** `{vector1.astype(int).tolist()}`"))
        display(Markdown(f"**Document 2 vector:** `{vector2.astype(int).tolist()}`"))

        display(Markdown("### Numerical calculation"))

        if np.isnan(similarity):
            display(Markdown(
                """
One of the documents has no tokens, so one vector has length 0.

Cosine similarity is undefined when either vector has norm 0.
"""
            ))
        else:
            display(Markdown(f"""
Dot product:

$$
\\mathbf{{a}} \\cdot \\mathbf{{b}} = {dot_product:.4f}
$$

Vector lengths:

$$
\\|\\mathbf{{a}}\\| = {norm1:.4f}
$$

$$
\\|\\mathbf{{b}}\\| = {norm2:.4f}
$$

Cosine similarity:

$$
\\frac{{{dot_product:.4f}}}{{{norm1:.4f} \\times {norm2:.4f}}}
=
{similarity:.4f}
$$

Cosine distance:

$$
1 - {similarity:.4f} = {distance:.4f}
$$
"""))


doc1_box.observe(update_similarity, names="value")
doc2_box.observe(update_similarity, names="value")


display(VBox([
    HBox([doc1_box, doc2_box]),
    output
]))

update_similarity()

In [ ]:
# Cat Sat Green Mat
# Cat Sat Rug
1 - (1 * 1 + 1 * 1 + 1 * 0 + 0 * 1 + 1 * 0) / (np.sqrt(4) * np.sqrt(3))

In [ ]:
# Cat Sat Mat
# Cat Sat Rug
v1 = np.array([1, 1, 0, 1])
v2 = np.array([1, 1, 1, 0])
1 - v1 @ v2 / (np.sqrt(v1 @ v1) * np.sqrt(v2 @ v2))

In [ ]:
# Cat Sat Mat
# Cat Sat Mat
v1 = np.array([1, 1, 1])
v2 = np.array([1, 1, 1])
1 - v1 @ v2 / (np.sqrt(v1 @ v1) * np.sqrt(v2 @ v2))

# https://arogozhnikov.github.io/2016/06/24/gradient_boosting_explained.html

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor, plot_tree
from ipywidgets import interact, Dropdown, IntSlider, FloatSlider


# --------------------------------------------------
# 1. Make a small regression dataset
# --------------------------------------------------

def make_dataset(S=100, random_state=8):
    rng = np.random.default_rng(random_state)

    X = rng.normal(size=(S, 4))

    y = (
        3.0 * X[:, 0]
        - 2.0 * X[:, 1]
        + 1.5 * (X[:, 2] > 0).astype(float)
        + 0.8 * X[:, 3]
        + rng.normal(scale=0.8, size=S)
    )

    feature_names = ["x0", "x1", "x2", "x3"]

    return X, y, feature_names


X, y, feature_names = make_dataset(S=100, random_state=7)


# --------------------------------------------------
# 2. Fit one small regression tree
# --------------------------------------------------

def fit_small_tree(
    X_train,
    y_train,
    candidate_features,
    max_leaf_nodes=4,
    random_state=0
):
    """
    Fit a small regression tree using only candidate_features.

    max_leaf_nodes controls the approximate size of the tree.
    For a stump, use max_leaf_nodes = 2.
    For a slightly deeper tree, use max_leaf_nodes = 3 or 4.
    """

    X_subset = X_train[:, candidate_features]

    tree = DecisionTreeRegressor(
        max_leaf_nodes=max_leaf_nodes,
        random_state=random_state
    )

    tree.fit(X_subset, y_train)

    model = {
        "tree": tree,
        "candidate_features": np.array(candidate_features),
        "max_leaf_nodes": max_leaf_nodes
    }

    return model


def predict_small_tree(model, X):
    candidate_features = model["candidate_features"]
    tree = model["tree"]

    return tree.predict(X[:, candidate_features])


# --------------------------------------------------
# 3. Random forest using small trees
# --------------------------------------------------

def fit_random_forest_small_trees(
    X,
    y,
    N,
    max_leaf_nodes=4,
    random_state=1
):
    """
    Fit N independent randomized small trees.

    For each tree:
    - choose sqrt(4) = 2 candidate features randomly
    - bootstrap the samples
    - fit a small regression tree using only those 2 features
    """

    rng = np.random.default_rng(random_state)
    S, D = X.shape

    models = []

    for n in range(N):
        candidate_features = rng.choice(D, size=2, replace=False)
        bootstrap_indices = rng.choice(S, size=S, replace=True)

        model = fit_small_tree(
            X_train=X[bootstrap_indices],
            y_train=y[bootstrap_indices],
            candidate_features=candidate_features,
            max_leaf_nodes=max_leaf_nodes,
            random_state=random_state + n
        )

        model["model_type"] = "Random Forest"
        model["tree_number"] = n + 1

        models.append(model)

    return models


def predict_random_forest_small_trees(models, X):
    all_predictions = np.column_stack([
        predict_small_tree(model, X) for model in models
    ])

    final_prediction = all_predictions.mean(axis=1)

    return final_prediction, all_predictions


# --------------------------------------------------
# 4. Gradient boosting using small trees
# --------------------------------------------------

def fit_gradient_boost_small_trees(
    X,
    y,
    N,
    max_leaf_nodes=4,
    learning_rate=0.2,
    random_state=1
):
    """
    Fit N small trees sequentially.

    Initial prediction:
        y_hat_0 = mean(y)

    At step n:
        residual = y - current_prediction
        fit a small tree to the residual
        current_prediction = current_prediction + learning_rate * tree_prediction
    """

    current_prediction = np.full_like(y, y.mean(), dtype=float)

    models = []

    for n in range(N):
        residual = y - current_prediction

        candidate_features = np.arange(X.shape[1])

        model = fit_small_tree(
            X_train=X,
            y_train=residual,
            candidate_features=candidate_features,
            max_leaf_nodes=max_leaf_nodes,
            random_state=random_state + n
        )

        tree_prediction = predict_small_tree(model, X)

        current_prediction = current_prediction + learning_rate * tree_prediction

        model["model_type"] = "Gradient Boosting"
        model["tree_number"] = n + 1
        model["learning_rate"] = learning_rate
        model["residual_mean_before_fit"] = residual.mean()
        model["residual_rmse_before_fit"] = np.sqrt(np.mean(residual**2))

        models.append(model)

    initial_prediction = y.mean()

    return models, initial_prediction


def predict_gradient_boost_small_trees(models, X, initial_prediction):
    prediction = np.full(X.shape[0], initial_prediction, dtype=float)

    all_tree_predictions = []

    for model in models:
        tree_prediction = predict_small_tree(model, X)
        all_tree_predictions.append(tree_prediction)

        prediction = prediction + model["learning_rate"] * tree_prediction

    return prediction, np.column_stack(all_tree_predictions)


# --------------------------------------------------
# 5. Plot trees with more vertical room
# --------------------------------------------------

def plot_all_trees(models, feature_names):
    """
    Draw each small tree separately, stacked vertically.

    This takes more vertical space but is easier to read than
    squeezing many trees into one row.
    """

    N = len(models)

    fig_height = max(3.5 * N, 4)

    fig, axes = plt.subplots(
        N,
        1,
        figsize=(13, fig_height)
    )

    if N == 1:
        axes = [axes]

    for ax, model in zip(axes, models):
        tree = model["tree"]
        candidate_features = model["candidate_features"]

        local_feature_names = [
            feature_names[j] for j in candidate_features
        ]

        title = (
            f"Tree {model['tree_number']} | "
            f"{model['model_type']} | "
            f"candidate features: {', '.join(local_feature_names)} | "
            f"max leaves: {model['max_leaf_nodes']}"
        )

        if model["model_type"] == "Gradient Boosting":
            title += (
                f" | fitting residuals"
                f" | residual RMSE before fit: "
                f"{model['residual_rmse_before_fit']:.3f}"
            )

        plot_tree(
            tree,
            ax=ax,
            feature_names=local_feature_names,
            filled=True,
            rounded=True,
            impurity=False,
            proportion=False,
            precision=2
        )

        ax.set_title(title, fontsize=11)

    plt.tight_layout()
    plt.show()


def print_tree_summary(model, feature_names):
    tree = model["tree"]
    candidate_features = model["candidate_features"]
    local_feature_names = [feature_names[j] for j in candidate_features]

    print(
        f"Tree {model['tree_number']} "
        f"({model['model_type']}), "
        f"candidate features: {local_feature_names}"
    )

    print(f"Number of leaves actually used: {tree.get_n_leaves()}")
    print(f"Tree depth actually used: {tree.get_depth()}")

    if model["model_type"] == "Gradient Boosting":
        print(
            f"Residual RMSE before fitting this tree: "
            f"{model['residual_rmse_before_fit']:.3f}"
        )

    print()


# --------------------------------------------------
# 6. Main interactive demo
# --------------------------------------------------

def ensemble_tree_demo(
    model_type="Random Forest",
    N=5,
    max_leaf_nodes=4,
    learning_rate=0.2
):
    if model_type == "Random Forest":
        models = fit_random_forest_small_trees(
            X,
            y,
            N=N,
            max_leaf_nodes=max_leaf_nodes,
            random_state=3
        )

        final_prediction, all_predictions = predict_random_forest_small_trees(
            models,
            X
        )

        formula = (
            "Random forest prediction:\n"
            "Each small tree independently predicts the target.\n\n"
            "Each tree uses:\n"
            "    - a bootstrap sample of the rows\n"
            "    - a random subset of sqrt(4) = 2 features\n"
            f"    - at most {max_leaf_nodes} leaf nodes\n\n"
            "Final prediction:\n"
            "    y_hat = average(tree_1 prediction, ..., tree_N prediction)"
        )

    else:
        models, initial_prediction = fit_gradient_boost_small_trees(
            X,
            y,
            N=N,
            max_leaf_nodes=max_leaf_nodes,
            learning_rate=learning_rate,
            random_state=3
        )

        final_prediction, all_predictions = predict_gradient_boost_small_trees(
            models,
            X,
            initial_prediction=initial_prediction
        )

        formula = (
            "Gradient boosting prediction:\n"
            "Initial prediction:\n"
            "    y_hat_0 = mean(y)\n\n"
            "At each step n:\n"
            "    residual_n = y - y_hat_current\n"
            "    tree_n fits residual_n\n"
            "    y_hat_current = y_hat_current + learning_rate * tree_n prediction\n\n"
            f"Each tree has at most {max_leaf_nodes} leaf nodes.\n\n"
            "Final prediction:\n"
            "    y_hat = mean(y) + learning_rate * sum(tree_n predictions)"
        )

    rmse = np.sqrt(np.mean((y - final_prediction) ** 2))

    print("=" * 90)
    print(f"Model type: {model_type}")
    print(f"Number of trees N: {N}")
    print(f"Maximum leaf nodes per tree: {max_leaf_nodes}")
    if model_type == "Gradient Boosting":
        print(f"Learning rate: {learning_rate}")
    print(f"Training RMSE: {rmse:.3f}")
    print("=" * 90)
    print()
    print(formula)
    print()

    print("=" * 90)
    print("Tree summaries")
    print("=" * 90)

    for model in models:
        print_tree_summary(model, feature_names)

    plot_all_trees(models, feature_names)

    print("=" * 90)
    print("First 10 target values and predictions")
    print("=" * 90)

    for i in range(10):
        print(
            f"sample {i:2d}: "
            f"true y = {y[i]:7.3f}, "
            f"predicted y = {final_prediction[i]:7.3f}"
        )

    plt.figure(figsize=(7, 6))
    plt.scatter(
        y,
        final_prediction,
        s=60,
        edgecolor="black",
        alpha=0.8
    )

    low = min(y.min(), final_prediction.min())
    high = max(y.max(), final_prediction.max())

    plt.plot(
        [low, high],
        [low, high],
        "r--",
        label="perfect prediction"
    )

    plt.xlabel("True target")
    plt.ylabel("Predicted target")
    plt.title(
        f"{model_type} with {N} Small Trees\n"
        f"max leaves = {max_leaf_nodes}, RMSE = {rmse:.3f}"
    )
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


# --------------------------------------------------
# 7. Jupyter controls
# --------------------------------------------------

interact(
    ensemble_tree_demo,
    model_type=Dropdown(
        options=["Random Forest", "Gradient Boosting"],
        value="Random Forest",
        description="Model"
    ),
    N=IntSlider(
        value=4,
        min=1,
        max=10,
        step=1,
        description="Trees"
    ),
    max_leaf_nodes=IntSlider(
        value=4,
        min=2,
        max=8,
        step=1,
        description="Max leaves"
    ),
    learning_rate=FloatSlider(
        value=0.2,
        min=0.05,
        max=1.0,
        step=0.05,
        description="Learn rate"
    )
)

In [ ]:
"""
Where to put this:
Paste this entire cell after your notebook's imports and run it as a new cell.
It is self-contained and does not replace or depend on your existing ensemble demo.
Use the Stage slider to watch each tree correct the model's current residuals.
"""

import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from ipywidgets import interact, IntSlider, FloatSlider


def staged_gradient_boosting_demo(
    stage=10,
    learning_rate=0.2,
    max_leaf_nodes=3,
    noise=0.35
):
    # Recreate the same dataset whenever a control changes.
    rng = np.random.default_rng(12)

    n_samples = 150
    x = rng.uniform(-3.0, 3.0, size=n_samples)
    true_function = lambda z: np.sin(1.5 * z) + 0.25 * z
    y = true_function(x) + rng.normal(0.0, noise, size=n_samples)

    # Use a fixed train/test split to make changes across slider settings
    # easier to interpret.
    permutation = rng.permutation(n_samples)
    n_train = int(0.7 * n_samples)
    train_indices = permutation[:n_train]
    test_indices = permutation[n_train:]

    x_train = x[train_indices]
    y_train = y[train_indices]
    x_test = x[test_indices]
    y_test = y[test_indices]

    grid = np.linspace(-3.0, 3.0, 500)
    X_train = x_train.reshape(-1, 1)
    X_test = x_test.reshape(-1, 1)
    X_grid = grid.reshape(-1, 1)

    # Gradient boosting begins with a constant prediction: the training mean.
    initial_prediction = y_train.mean()
    prediction_train = np.full(len(y_train), initial_prediction)
    prediction_test = np.full(len(y_test), initial_prediction)
    prediction_grid = np.full(len(grid), initial_prediction)

    train_predictions_by_stage = [prediction_train.copy()]
    test_predictions_by_stage = [prediction_test.copy()]
    grid_predictions_by_stage = [prediction_grid.copy()]
    residuals_before_each_stage = []
    corrections_by_stage = []

    # Fit each small tree to the residuals left by the current ensemble.
    for _ in range(stage):
        residuals = y_train - prediction_train
        tree = DecisionTreeRegressor(
            max_leaf_nodes=max_leaf_nodes,
            random_state=0
        )
        tree.fit(X_train, residuals)

        tree_train = tree.predict(X_train)
        tree_test = tree.predict(X_test)
        tree_grid = tree.predict(X_grid)

        residuals_before_each_stage.append(residuals.copy())
        corrections_by_stage.append(learning_rate * tree_grid)

        prediction_train += learning_rate * tree_train
        prediction_test += learning_rate * tree_test
        prediction_grid += learning_rate * tree_grid

        train_predictions_by_stage.append(prediction_train.copy())
        test_predictions_by_stage.append(prediction_test.copy())
        grid_predictions_by_stage.append(prediction_grid.copy())

    selected_stage = min(stage, len(grid_predictions_by_stage) - 1)

    train_prediction = train_predictions_by_stage[selected_stage]
    test_prediction = test_predictions_by_stage[selected_stage]
    fitted_curve = grid_predictions_by_stage[selected_stage]

    train_rmse = np.sqrt(np.mean((y_train - train_prediction) ** 2))
    test_rmse = np.sqrt(np.mean((y_test - test_prediction) ** 2))

    fig, (ax_fit, ax_correction) = plt.subplots(
        2,
        1,
        figsize=(11, 8),
        gridspec_kw={"height_ratios": [2, 1]},
        sharex=True
    )

    # Upper panel: current boosted model and data.
    ax_fit.plot(
        grid,
        true_function(grid),
        color="black",
        linestyle=":",
        linewidth=2,
        label="True underlying function"
    )
    ax_fit.plot(
        grid,
        fitted_curve,
        color="#D62728",
        linewidth=3,
        label=f"Boosted model after stage {selected_stage}"
    )
    ax_fit.scatter(
        x_train,
        y_train,
        color="#2878B5",
        edgecolor="white",
        linewidth=0.6,
        s=45,
        alpha=0.85,
        label="Training data"
    )
    ax_fit.scatter(
        x_test,
        y_test,
        color="#E07A1F",
        edgecolor="white",
        linewidth=0.6,
        s=55,
        marker="s",
        alpha=0.9,
        label="Held-out test data"
    )
    ax_fit.set_ylabel("Target")
    ax_fit.set_title(
        f"Gradient boosting: {selected_stage} correction stages | "
        f"learning rate={learning_rate:.2f} | "
        f"max leaves/tree={max_leaf_nodes}\n"
        f"Train RMSE={train_rmse:.3f} | Test RMSE={test_rmse:.3f}"
    )
    ax_fit.grid(alpha=0.25)
    ax_fit.legend(loc="upper left", ncols=2)

    # Lower panel: the residuals before the selected tree and the correction
    # that tree adds to the ensemble.
    if selected_stage == 0:
        residuals_before = y_train - initial_prediction
        correction_curve = np.zeros_like(grid)
        correction_label = "No tree added yet"
    else:
        residuals_before = residuals_before_each_stage[selected_stage - 1]
        correction_curve = corrections_by_stage[selected_stage - 1]
        correction_label = (
            f"Stage {selected_stage} correction "
            f"(learning rate × tree prediction)"
        )

    ax_correction.scatter(
        x_train,
        residuals_before,
        color="#2878B5",
        edgecolor="white",
        linewidth=0.5,
        s=42,
        alpha=0.8,
        label="Residuals before this stage"
    )
    ax_correction.plot(
        grid,
        correction_curve,
        color="#2CA02C",
        linewidth=3,
        drawstyle="steps-post",
        label=correction_label
    )
    ax_correction.axhline(0, color="black", linewidth=1, alpha=0.6)
    ax_correction.set_xlabel("x")
    ax_correction.set_ylabel("Residual / correction")
    ax_correction.set_title(
        "Each small tree is fitted to the current training residuals"
    )
    ax_correction.grid(alpha=0.25)
    ax_correction.legend(loc="best")

    plt.tight_layout()
    plt.show()


interact(
    staged_gradient_boosting_demo,
    stage=IntSlider(
        value=10,
        min=0,
        max=50,
        step=1,
        description="Stage"
    ),
    learning_rate=FloatSlider(
        value=0.2,
        min=0.05,
        max=1.0,
        step=0.05,
        description="Learn rate"
    ),
    max_leaf_nodes=IntSlider(
        value=3,
        min=2,
        max=8,
        step=1,
        description="Max leaves"
    ),
    noise=FloatSlider(
        value=0.35,
        min=0.05,
        max=0.8,
        step=0.05,
        description="Noise"
    )
);